In [2]:
import numpy as np
import matplotlib.pyplot as plt
from public_tests import *

%matplotlib inline

/Users/krishd/Documents/Decision trees/public_tests.py:72: SyntaxWarning: invalid escape sequence '\{'
  assert np.allclose(right, expected['right']) and np.allclose(left, expected['left']), f"Wrong value when target is at index 0. \nExpected: {expected} \ngot: \{left:{left}, 'right': {right}\}"
/Users/krishd/Documents/Decision trees/public_tests.py:72: SyntaxWarning: invalid escape sequence '\}'
  assert np.allclose(right, expected['right']) and np.allclose(left, expected['left']), f"Wrong value when target is at index 0. \nExpected: {expected} \ngot: \{left:{left}, 'right': {right}\}"


In [3]:
X_train = np.array([[1,1,1],[1,0,1],[1,0,0],[1,0,0],[1,1,1],[0,1,1],[0,0,0],[1,0,1],[0,1,0],[1,0,0]])
y_train = np.array([1,1,0,0,1,0,0,1,1,0])

In [5]:
print("First few elements of X_train:\n", X_train[:5])
print("Type of X_train:",type(X_train))

print("First few elements of y_train:", y_train[:5])
print("Type of y_train:",type(y_train))

print ('The shape of X_train is:', X_train.shape)
print ('The shape of y_train is: ', y_train.shape)
print ('Number of training examples (m):', len(X_train))

First few elements of X_train:
 [[1 1 1]
 [1 0 1]
 [1 0 0]
 [1 0 0]
 [1 1 1]]
Type of X_train: <class 'numpy.ndarray'>
First few elements of y_train: [1 1 0 0 1]
Type of y_train: <class 'numpy.ndarray'>
The shape of X_train is: (10, 3)
The shape of y_train is:  (10,)
Number of training examples (m): 10


In [27]:
def compute_entropy(y):
    if len(y) == 0:
        return 0
    
    p1 = sum(y) / len(y)
    p0 = 1 - p1
    if p1 == 0 or p0 == 0:
        return 0
    entropy = -p1*np.log2(p1) - p0*np.log2(p0)
    return entropy

In [ ]:
# Compute entropy at the root node (i.e. with all examples)
# Since we have 5 edible and 5 non-edible mushrooms, the entropy should be 1"

print("Entropy at root node: ", compute_entropy(y_train)) 

# UNIT TESTS
compute_entropy_test(compute_entropy)

Entropy at root node:  1.0
 All tests passed.


In [29]:
def split_dataset(X, node_indices, feature):
    """
    Splits the data based on a given feature at a given node.
    Returns:
        left_indices: indices where feature value is 0
        right_indices: indices where feature value is 1
    """
    left_indices = []
    right_indices = []
    
    for i in node_indices:
        if X[i][feature] == 0:
            left_indices.append(i)
        else:
            right_indices.append(i)
            
    return left_indices, right_indices

In [ ]:
def compute_information_gain(X, y, node_indices, feature):
    
    
    # Calls a helper function to divide the current row indices into a left group (feature value 0) and a right group (feature value 1).
    left_indices, right_indices = split_dataset(X, node_indices, feature)

# Extracts the feature values (X_node) and target labels (y_node) for all rows currently at this parent node.
    X_node, y_node = X[node_indices], y[node_indices] #x[3] will return like 3rd row of data points

# Pulls out the feature data (X_left) and target labels (y_left) for the rows that went down the left branch.
    X_left, y_left = X[left_indices], y[left_indices]
#goes everywherw and collects feature that has 0

# Pulls out the feature data (X_right) and target labels (y_right) for the rows that went down the right branch.
    X_right, y_right = X[right_indices], y[right_indices]
#goes everywherw and collects feature that has 1

# Computes the entropy of the parent node (H_parent) using the target labels (y_node).

    H_parent = compute_entropy(y_node)
    w_left = len(y_left) / len(y_node)
    w_right = len(y_right) / len(y_node)
    H_children = (w_left * compute_entropy(y_left)) + (w_right * compute_entropy(y_right))
    
    # Information gain is the parent entropy minus the weighted children entropy
    information_gain = H_parent - H_children
    
    return information_gain
   

    

In [33]:
root_indices = list(range(len(X_train)))
info_gain0 = compute_information_gain(X_train, y_train, root_indices, feature=0)
print("Information Gain from splitting the root on brown cap: ", info_gain0)
    
info_gain1 = compute_information_gain(X_train, y_train, root_indices, feature=1)
print("Information Gain from splitting the root on tapering stalk shape: ", info_gain1)

info_gain2 = compute_information_gain(X_train, y_train, root_indices, feature=2)
print("Information Gain from splitting the root on solitary: ", info_gain2)

# UNIT TESTS
compute_information_gain_test(compute_information_gain)

Information Gain from splitting the root on brown cap:  0.034851554559677034
Information Gain from splitting the root on tapering stalk shape:  0.12451124978365313
Information Gain from splitting the root on solitary:  0.2780719051126377
 All tests passed.


In [ ]:
def get_best_split(X, y, node_indices):   

    # Some useful variables
    num_features = X.shape[1]

    # You need to return the following variables correctly
    best_feature = -1

    ### START CODE HERE ###
    max_info_gain = 0

    # Iterate through all features
    for feature in range(num_features): 
        
        # Compute the information gain from splitting on this feature
        info_gain = compute_information_gain(X, y, node_indices, feature)
        
        # If the information gain is larger than the max seen so far
        if info_gain > max_info_gain:  
            # Set the max_info_gain and best_feature
            max_info_gain = info_gain
            best_feature = feature
    ### END CODE HERE ##    

    return best_feature

In [41]:
best_feature = get_best_split(X_train, y_train, root_indices)
print("Best feature to split on: %d" % best_feature)

# UNIT TESTS
get_best_split_test(get_best_split)

Best feature to split on: 2
 All tests passed.


In [44]:
tree = []

def build_tree_recursive(X, y, node_indices, branch_name, max_depth, current_depth):
    """
    Build a tree using the recursive algorithm that split the dataset into 2 subgroups at each node.
    This function just prints the tree.
    
    Args:
        X (ndarray):            Data matrix of shape(n_samples, n_features)
        y (array like):         list or ndarray with n_samples containing the target variable
        node_indices (ndarray): List containing the active indices. I.e, the samples being considered in this step.
        branch_name (string):   Name of the branch. ['Root', 'Left', 'Right']
        max_depth (int):        Max depth of the resulting tree. 
        current_depth (int):    Current depth. Parameter used during recursive call.
   
    """ 

    # Maximum depth reached - stop splitting
    if current_depth == max_depth:
        formatting = " "*current_depth + "-"*current_depth
        print(formatting, "%s leaf node with indices" % branch_name, node_indices)
        return
   
    # Otherwise, get best split and split the data
    # Get the best feature and threshold at this node
    best_feature = get_best_split(X, y, node_indices) 
    tree.append((current_depth, branch_name, best_feature, node_indices))
    
    formatting = "-"*current_depth
    print("%s Depth %d, %s: Split on feature: %d" % (formatting, current_depth, branch_name, best_feature))
    
    # Split the dataset at the best feature
    left_indices, right_indices = split_dataset(X, node_indices, best_feature)
    
    # continue splitting the left and the right child. Increment current depth
    build_tree_recursive(X, y, left_indices, "Left", max_depth, current_depth+1)
    build_tree_recursive(X, y, right_indices, "Right", max_depth, current_depth+1)

In [45]:
build_tree_recursive(X_train, y_train, root_indices, "Root", max_depth=2, current_depth=0)


 Depth 0, Root: Split on feature: 2
- Depth 1, Left: Split on feature: 1
  -- Left leaf node with indices [2, 3, 6, 9]
  -- Right leaf node with indices [8]
- Depth 1, Right: Split on feature: 0
  -- Left leaf node with indices [5]
  -- Right leaf node with indices [0, 1, 4, 7]
